In [7]:
a = 2.0
b = -3.0
c = 10.0

d = a * b
e = d + c
f = e * e

print(d, e, f)

-6.0 4.0 16.0


d = -6.0
e = 4.0
f = 16.0

## What the backward pass is doing

A **backward pass** is bookkeeping for sensitivities. A sensitivity asks: if I nudge one value a tiny bit, how much does the final output change?

For the local step `f = e * e`, start from `e = 4.0`, so `f = 16.0`. If we nudge `e` to `4.001`, then `f` becomes:

```python
4.001 * 4.001 = 16.008001
```

So the actual change in `e` is:

```python
4.001 - 4.0 = 0.001
```

And the actual change in `f` is:

```python
16.008001 - 16.0 = 0.008001
```

The sensitivity is the ratio between those two changes:

```python
change_in_f / change_in_e = 0.008001 / 0.001 = 8.001
```

That is why we say: near `e = 4.0`, `f` changes about `8` times as much as `e`. The exact local derivative for `f = e * e` is `df/de = 2 * e`, so at `e = 4.0`, `df/de = 8.0`.


In [8]:
# Numerical nudge check for df/de when f = e * e

e = 4.0
h = 0.001

old_f = e * e
new_e = e + h
new_f = new_e * new_e

change_in_e = new_e - e
change_in_f = new_f - old_f
numerical_df_de = change_in_f / change_in_e
exact_df_de = 2.0 * e

print(f"old_f = {old_f:.6f}")
print(f"new_f = {new_f:.6f}")
print(f"change_in_e = {change_in_e:.6f}")
print(f"change_in_f = {change_in_f:.6f}")
print(f"numerical df/de = {numerical_df_de:.6f}")
print(f"exact df/de = {exact_df_de:.6f}")

old_f = 16.000000
new_f = 16.008001
change_in_e = 0.001000
change_in_f = 0.008001
numerical df/de = 8.001000
exact df/de = 8.000000


## Backward pass through addition: `e = d + c`

Now walk one step earlier. We already found that the final output cares about `e` with sensitivity:

```python
df/de = 8.0
```

For the local step `e = d + c`, change one input at a time and hold the other fixed.

If `d` increases by `1.0` while `c` stays at `10.0`, then `e` also increases by `1.0`:

```python
old_d = -6.0
c = 10.0
old_e = old_d + c      # 4.0

new_d = -5.0           # d increased by 1.0
new_e = new_d + c      # 5.0

de/dd = (new_e - old_e) / (new_d - old_d)
      = 1.0 / 1.0
      = 1.0
```

The same idea works for `c`. If `c` increases by `1.0` while `d` stays fixed, then `e` also increases by `1.0`:

```python
de/dc = 1.0
```

Now multiply by the upstream sensitivity from `f` to `e`:

```python
df/dd = df/de * de/dd = 8.0 * 1.0 = 8.0
df/dc = df/de * de/dc = 8.0 * 1.0 = 8.0
```

So, near these values, both `d` and `c` affect `f` with sensitivity `8.0`.


In [9]:
# Code version of the backward pass through e = d + c

d = -6.0
c = 10.0
h = 0.001

old_e = d + c
new_e_from_d = (d + h) + c
new_e_from_c = d + (c + h)

de_dd = (new_e_from_d - old_e) / h
de_dc = (new_e_from_c - old_e) / h

df_de = 8.0

df_dd = df_de * de_dd
df_dc = df_de * de_dc

print(f"de/dd = {de_dd:.6f}")
print(f"de/dc = {de_dc:.6f}")
print(f"df/dd = {df_dd:.6f}")
print(f"df/dc = {df_dc:.6f}")

de/dd = 1.000000
de/dc = 1.000000
df/dd = 8.000000
df/dc = 8.000000


## Backward pass through multiplication: `d = a * b`

Now walk one more step earlier. We know the final output cares about `d` with sensitivity:

```python
df/dd = 8.0
```

For the local step `d = a * b`, change one input at a time and hold the other fixed.

If `a` changes while `b` stays fixed, the local sensitivity is `b`:

```python
a = 2.0
b = -3.0
d = a * b          # -6.0

new_a = 2.001
new_d = new_a * b  # -6.003

change_in_a = 0.001
change_in_d = -0.003

dd/da = change_in_d / change_in_a
      = -0.003 / 0.001
      = -3.0
      = b
```

Now chain that local sensitivity with the upstream sensitivity:

```python
df/da = df/dd * dd/da
      = 8.0 * -3.0
      = -24.0
```

The negative sign means that increasing `a` makes `d` smaller here, because `b` is negative, and that makes the final output `f` smaller near these values.


In [10]:
# Code version of the backward pass through d = a * b for a

a = 2.0
b = -3.0
h = 0.001

old_d = a * b
new_d_from_a = (a + h) * b

dd_da = (new_d_from_a - old_d) / h

df_dd = 8.0
df_da = df_dd * dd_da

print(f"old_d = {old_d:.6f}")
print(f"new_d_from_a = {new_d_from_a:.6f}")
print(f"dd/da = {dd_da:.6f}")
print(f"df/da = {df_da:.6f}")

old_d = -6.000000
new_d_from_a = -6.003000
dd/da = -3.000000
df/da = -24.000000


## Checkpoint: fill in one missing local derivative

For the same local step:

```python
d = a * b
```

If `b` changes and `a` stays fixed, fill in the missing local derivative:

```python
dd/db = a
```

Then chain it with `df/dd = 8.0`:

```python
df/db = df/dd * dd/db
      = 8.0 * 2.0
      = 16.0
```


In [12]:
# Checkpoint code cell: fill in dd_db before running the final print.
# Hint: for d = a * b, change b and hold a fixed.

a = 2.0
b = -3.0
df_dd = 8.0

# Replace None with the local derivative dd/db.
dd_db = a

if dd_db is None:
    print("Fill in dd_db first, then rerun this cell.")
else:
    df_db = df_dd * dd_db
    print(f"dd/db = {dd_db:.6f}")
    print(f"df/db = {df_db:.6f}")

dd/db = 2.000000
df/db = 16.000000
